# Notebook 3 — Verb Extraction & Caption Masking

Extracts the main action verb from each caption using a two-pass approach
(spaCy dependency parse → HuggingFace POS fallback), builds a closed verb vocabulary,
masks the verb in each caption, and writes the final labeled dataset.

**Why we read `segments.csv` here (not `segments_available.csv`):**
Verb extraction is purely text-based — no video files are opened.
Using all segments gives a richer vocabulary. The video availability filter
is applied in NB4 when video files are actually needed.

**Outputs:**
- `processed/verb_vocab.json` — verb → label_id mapping (114 classes)
- `processed/segments_labeled.csv` — segments with `masked_caption` and `verb_label`

**Note for NB4:** Before opening any video file, NB4 merges `segments_labeled.csv`
with `segments_available.csv` on `video_id` to drop rows whose video is not on disk.


> **⚠️ DEPRECATED — do not use, and don't run this notebook.**
>
> This is a second, different "Notebook 3" that exists alongside
> `notebooks/03_verb_extraction.ipynb`, which is the real one — every downstream
> notebook (NB4/4b/5/5b/6/8) reads its output (`segments_verbs.csv`, later
> `segments_verbs_timed(_clean).csv`), and its numbers match the project's reported
> stats exactly (12,572 ROOT + 457 fallback = 13,029 rows).
>
> This notebook is unrelated dead code, discovered during a repo audit on
> 2026-08-26:
> - Its first cell already fails — `FileNotFoundError` on
>   `/home/ubuntu/youcook2_project/paths.json`, a path that doesn't exist anywhere
>   in this project (the real `paths.json` lives at the project root).
> - It uses a completely different, incompatible methodology: a HuggingFace BERT
>   POS-tagger fallback (not the auxiliary-filtering fallback the real NB3 uses),
>   a frequency-capped **114-class** vocabulary (`MIN_FREQ=5, MAX_VOCAB=150`),
>   and writes to `processed/verb_vocab.json` / `processed/segments_labeled.csv` —
>   none of which anything downstream reads. The real pipeline uses an open,
>   **282-verb** vocabulary with no frequency cutoff.
> - It was renamed with a `DEPRECATED_` prefix and moved out of `notebooks/` to
>   avoid being mistaken for the real one. Kept only for history — safe to delete
>   entirely if it isn't needed.

In [1]:
import os, json, pandas as pd

with open('/home/ubuntu/youcook2_project/paths.json') as f:
    PATHS = json.load(f)

# Read ALL segments — text only, no video needed
df = pd.read_csv(os.path.join(PATHS['annotations'], 'segments.csv'))
print(f"Loaded {len(df):,} total segments (text only)")
df.head(3)


FileNotFoundError: [Errno 2] No such file or directory: '/home/ubuntu/youcook2_project/paths.json'

## Pass 1 — spaCy dependency parsing
The dependency parser identifies the ROOT token of each sentence.
In cooking instructions the ROOT is almost always the imperative verb ("slice", "pour", "stir").
We lemmatise so "slicing" and "sliced" both map to "slice".

Strategy: ROOT if it is a VERB or AUX, else fall back to the first VERB in the sentence.


In [ ]:
import spacy
nlp = spacy.load('en_core_web_sm')

def extract_verb_spacy(caption: str):
    """
    Returns (verb_text, verb_lemma, char_start, char_end) or None.
    """
    doc = nlp(caption)
    for token in doc:
        if token.dep_ == 'ROOT' and token.pos_ in ('VERB', 'AUX'):
            return token.text, token.lemma_.lower(), token.idx, token.idx + len(token.text)
    for token in doc:
        if token.pos_ == 'VERB':
            return token.text, token.lemma_.lower(), token.idx, token.idx + len(token.text)
    return None

# Sanity check
for cap in ["Slice the onions thinly", "Pour milk into the bowl",
            "add salt and pepper", "Stir fry for two minutes"]:
    r = extract_verb_spacy(cap)
    print(f"  '{cap:<35s}' → {r[1] if r else 'NONE'}")


## Pass 2 — HuggingFace POS fallback
For captions where spaCy finds no verb (fragment annotations, unusual phrasing),
a BERT-based POS tagger fills the gap.

The HF_TOKEN set in NB1 is used automatically for authenticated access.


In [ ]:
import os
from transformers import pipeline

# Confirm token is set before loading the model
hf_tok = os.environ.get('HF_TOKEN', '')
if not hf_tok.startswith('hf_'):
    print("⚠️  HF_TOKEN not set — re-run the token cell in NB1 or set os.environ['HF_TOKEN'] here.")
else:
    print(f"HF_TOKEN present ✓")

pos_pipe = pipeline(
    'token-classification',
    model='vblagoje/bert-english-uncased-finetuned-pos',
    aggregation_strategy='simple',
    device=0,  # GPU for speed
    token=hf_tok if hf_tok else None,
)

VERB_TAGS = {'VB', 'VBZ', 'VBP', 'VBD', 'VBN', 'VBG'}

def extract_verb_hf(caption: str):
    """Fallback: first VERB tag from the HF POS model."""
    try:
        for tag in pos_pipe(caption):
            if tag['entity_group'] in VERB_TAGS:
                word  = tag['word']
                start = caption.lower().find(word.lower())
                if start >= 0:
                    return word, word.lower(), start, start + len(word)
    except Exception:
        pass
    return None

print("HF fallback ready ✓")
# Sanity checks — both should return a result
print(f"  'chop finely'   → {extract_verb_hf('chop finely')}")
print(f"  'into the bowl' → {extract_verb_hf('into the bowl')}  (expect None — no verb)")


## Run two-pass extraction across all segments


In [ ]:
from tqdm.notebook import tqdm

records   = []
n_spacy, n_hf, n_none = 0, 0, 0

for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting verbs"):
    caption = row['caption']
    result  = extract_verb_spacy(caption)
    source  = 'spacy'

    if result is None:
        result = extract_verb_hf(caption)
        source = 'hf' if result else 'none'

    if result:
        verb_text, verb_lemma, v_start, v_end = result
        if source == 'spacy': n_spacy += 1
        else:                 n_hf    += 1
    else:
        verb_text = verb_lemma = v_start = v_end = None
        n_none += 1

    records.append({**row.to_dict(),
                    'verb_text':   verb_text,
                    'verb_lemma':  verb_lemma,
                    'verb_start':  v_start,
                    'verb_end':    v_end,
                    'verb_source': source})

df_verbs = pd.DataFrame(records)
print(f"spaCy found  : {n_spacy:,}")
print(f"HF fallback  : {n_hf:,}")
print(f"No verb found: {n_none:,}")
print(f"Coverage     : {100*(1 - n_none/len(df)):.1f}%")

# Inspect unresolved captions — useful for diagnosing gaps
unresolved = df_verbs[df_verbs['verb_lemma'].isna()]
if len(unresolved) > 0:
    print(f"\nSample unresolved captions ({len(unresolved)} total):")
    for cap in unresolved['caption'].sample(min(10, len(unresolved))):
        print(f"  '{cap}'")


## Build verb vocabulary
Keep verbs appearing at least `MIN_FREQ` times, cap at `MAX_VOCAB` classes.
Verbs below MIN_FREQ have too few examples for reliable training or evaluation.


In [ ]:
from collections import Counter

MIN_FREQ  = 5
MAX_VOCAB = 150

df_found = df_verbs[df_verbs['verb_lemma'].notna()].copy()
counts   = Counter(df_found['verb_lemma'])

vocab_items = sorted(
    [(v, c) for v, c in counts.items() if c >= MIN_FREQ],
    key=lambda x: -x[1]
)[:MAX_VOCAB]

verb_to_id = {v: i for i, (v, _) in enumerate(vocab_items)}
id_to_verb = {str(i): v for v, i in verb_to_id.items()}  # JSON keys must be str

vocab_path = os.path.join(PATHS['processed'], 'verb_vocab.json')
with open(vocab_path, 'w') as f:
    json.dump({'verb_to_id': verb_to_id,
               'id_to_verb': id_to_verb,
               'counts': dict(vocab_items)}, f, indent=2)

print(f"Vocabulary: {len(verb_to_id)} verbs  (MIN_FREQ={MIN_FREQ}, MAX_VOCAB={MAX_VOCAB})")
print(f"Saved → {vocab_path}")
print(f"\nTop 20 verbs:")
for v, c in vocab_items[:20]:
    print(f"  {v:<20s} {c:4d}")

# Class imbalance warning — important for NB5 weighted loss
top_count    = vocab_items[0][1]
second_count = vocab_items[1][1]
print(f"\n⚠️  Class imbalance:")
print(f"   '{vocab_items[0][0]}' (label 0): {top_count} examples")
print(f"   '{vocab_items[1][0]}' (label 1): {second_count} examples")
print(f"   Ratio: {top_count/second_count:.1f}x")
print(f"   → NB5 uses class-weighted CrossEntropy loss to correct for this")


## Mask captions and assign label IDs

The verb's exact surface form is replaced with `[MASK]` at the character position
found by the extractor — word boundaries are always respected.

**Note on `[MASK]` and the DinoTXT tokenizer:**
The tokenizer used in NB4/NB5 is CLIP's SimpleTokenizer — it has NO `[MASK]` special token.
The string `[MASK]` is encoded as regular BPE subwords. This is fine because the DinoTXT
TextTower uses 'first' token pooling (start-of-text token), not position-based mask pooling.
The surrounding context of the masked caption is what informs the text embedding.


In [ ]:
def mask_caption(caption, verb_text, v_start, v_end):
    return caption[:v_start] + '[MASK]' + caption[v_end:]

labeled = []
n_in, n_out = 0, 0

for _, row in df_found.iterrows():
    lemma = row['verb_lemma']
    if lemma not in verb_to_id:
        n_out += 1
        continue
    masked = mask_caption(row['caption'], row['verb_text'],
                          int(row['verb_start']), int(row['verb_end']))
    n_in += 1
    labeled.append({
        'segment_id':     row['segment_id'],
        'video_id':       row['video_id'],
        'subset':         row['subset'],
        'recipe_type':    row['recipe_type'],
        'start_sec':      row['start_sec'],
        'end_sec':        row['end_sec'],
        'duration':       row['duration'],
        'caption':        row['caption'],
        'masked_caption': masked,
        'verb_lemma':     lemma,
        'verb_label':     verb_to_id[lemma],
    })

df_labeled = pd.DataFrame(labeled)

print(f"In-vocabulary  : {n_in:,}")
print(f"Out-of-vocab   : {n_out:,}  (dropped — too rare to train/eval reliably)")
print(f"\nSample:")
for _, r in df_labeled.sample(4).iterrows():
    print(f"  {r['caption']}")
    print(f"  → {r['masked_caption']}  [label {r['verb_label']}: {r['verb_lemma']}]")
    print()


In [ ]:
train = df_labeled[df_labeled['subset'] == 'training']
val   = df_labeled[df_labeled['subset'] == 'validation']
print(f"Train: {len(train):,}  |  Val: {len(val):,}  |  Classes: {df_labeled['verb_label'].nunique()}")

labeled_path = os.path.join(PATHS['processed'], 'segments_labeled.csv')
df_labeled.to_csv(labeled_path, index=False)
print(f"Saved → {labeled_path}")
print(f"\nReminder for NB4: merge segments_labeled.csv with segments_available.csv")
print(f"on 'video_id' before opening any video files.")


## Done ✓
Proceed to **Notebook 4** to extract DinoTXT video embeddings.

**What was produced:**
- `processed/verb_vocab.json` — 114 verb classes with frequency counts
- `processed/segments_labeled.csv` — labeled segments with masked captions

**What NB4 must do before opening video files:**
Filter `segments_labeled.csv` against `segments_available.csv` on `video_id`.
